# 08.1 — Perfil de Menor Risco: Documentadas

**Purpose:** Data loading, documented vs non-documented comparison with p-values,
protective profile with adjusted OR, and protective combinations.

Outputs → `out/08.1_perfil_documentados/`
Artifacts (shared with 08.2) → `out/artifacts_menor_risco/`

In [ ]:
from __future__ import annotations
import sys
import pickle
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats as scipy_stats

warnings.filterwarnings("ignore")

ROOT    = Path(__file__).resolve().parent.parent
PARQUET = ROOT / "data" / "gestante_quadro_clinico.parquet"
OUT_DIR = ROOT / "out" / "08.1_perfil_documentados"
OUT_DIR.mkdir(parents=True, exist_ok=True)

ART_DIR = ROOT / "out" / "artifacts_menor_risco"
ART_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT / "src"))

# ── Idioma / Language ─────────────────────────────────────────────────────────
LANG = "pt"  # "pt" | "en"  ← única linha a alterar para mudar idioma

_T = {
    "pt": {
        "comp_doc_lbl":      "Documentadas",
        "comp_sem_lbl":      "Sem documentação",
        "comp_ylabel":       "Taxa de retorno 30d (%)",
        "comp_title":        "Retorno 30d: Documentadas × Não Documentadas por Segmento CID\n(* p<0.05  ** p<0.01  *** p<0.001)",
        "perfil_media_fmt":  "Média ({ref:.1f}%)",
        "perfil_xlabel":     "Taxa de retorno 30d (%)",
        "perfil_title":      "Perfil de Risco por Condição — Pacientes COM Documentação Clínica\n(ordenado por taxa de retorno; linha = média; * p<0.05)",
        "or_xlabel":         "Odds Ratio Ajustado (IC 95%)",
        "or_title":          "OR Ajustado — Condições Protetoras vs Retorno 30d\n(ajustado por idade, paridade, tipo atendimento, segmento CID)",
        "comb_media_fmt":    "Média ({ref:.1f}%)",
        "comb_xlabel":       "Taxa de retorno 30d (%)",
        "comb_title":        "Combinações de Condições Protetoras\n(Pacientes COM Documentação Clínica)",
    },
    "en": {
        "comp_doc_lbl":      "Documented",
        "comp_sem_lbl":      "Not documented",
        "comp_ylabel":       "30-day return rate (%)",
        "comp_title":        "30-day Return: Documented × Non-Documented by ICD-10 Segment\n(* p<0.05  ** p<0.01  *** p<0.001)",
        "perfil_media_fmt":  "Mean ({ref:.1f}%)",
        "perfil_xlabel":     "30-day return rate (%)",
        "perfil_title":      "Risk Profile by Condition — Patients WITH Clinical Documentation\n(sorted by return rate; line = mean; * p<0.05)",
        "or_xlabel":         "Adjusted Odds Ratio (95% CI)",
        "or_title":          "Adjusted OR — Protective Conditions vs 30-day Return\n(adjusted for age, parity, attendance type, ICD-10 segment)",
        "comb_media_fmt":    "Mean ({ref:.1f}%)",
        "comb_xlabel":       "30-day return rate (%)",
        "comb_title":        "Combinations of Protective Conditions\n(Patients WITH Clinical Documentation)",
    },
}
T = _T[LANG]

## Section 1 — Data Loading

In [ ]:
import duckdb as _ddb

_DUCKDB        = ROOT / "data" / "gestante.duckdb"
_PARQUET_ENRIQ = ROOT / "data" / "gestante_quadro_clinico_enriquecido.parquet"

_tabs = _ddb.connect(str(_DUCKDB)).execute("SHOW TABLES").df()["name"].tolist()
if "gestante_enriquecido" in _tabs:
    _con = _ddb.connect(str(_DUCKDB))
    df = _con.execute("SELECT * FROM gestante_enriquecido").df()
    _con.close()
    df.columns = df.columns.str.lower()
    print(f"DuckDB gestante_enriquecido carregado: {len(df):,} registros")
elif _PARQUET_ENRIQ.exists():
    df = pd.read_parquet(_PARQUET_ENRIQ)
    df.columns = df.columns.str.lower()
    print(f"Parquet enriquecido carregado: {len(df):,} registros")
else:
    # fallback: build from scratch
    from build_obstetric_features import enrich_obstetric
    df = pd.read_parquet(PARQUET)
    df.columns = df.columns.str.lower()
    df = df[df["is_recem_nascido"] == 0].copy()
    LLM_PARQUET = ROOT / "out" / "llm_extracted_20k.parquet"
    df = enrich_obstetric(df, llm_parquet=str(_DUCKDB))
    df = df[(df.flag_neonatal == 0)].copy()
    print(f"Processado do zero: {len(df):,} registros")

# Recompute datetime and derived columns after DuckDB replaces df
for _c in ["dt_atendimento", "dt_alta"]:
    if _c in df.columns:
        df[_c] = pd.to_datetime(df[_c], errors="coerce")
df["ano"] = df["dt_atendimento"].dt.year
df["mes"] = df["dt_atendimento"].dt.month

# Target: retorno 30d and 90d
df["y"]     = (df["sn_retorno_30d"].fillna("N") == "S").astype(int)
df["y_90d"] = (df["sn_retorno_90d"].fillna("N") == "S").astype(int)

# Documentation flags
df["tem_evolucao"] = df["qtd_evolucoes_documentadas"].fillna(0).gt(0).astype(int)
df["tem_queixa"]   = df["ds_queixa_principal"].notna().astype(int)
df["tem_qualquer"] = ((df["tem_evolucao"] == 1) | (df["tem_queixa"] == 1)).astype(int)
df["doc_rica"]     = (df["qtd_evolucoes_documentadas"].fillna(0) >= 3).astype(int)

# Numeric coercions
for c in ["semanas_gestacao", "idade_anos", "vl_escore_triagem",
          "nr_consultas_pre_natal", "duracao_horas_aprox",
          "media_peso_rn_g", "min_apgar_5min", "qtd_pre_med"]:
    df[c] = pd.to_numeric(df.get(c), errors="coerce")

# CID segment
def _seg(c):
    if pd.isna(c): return "SEM_CID"
    c = str(c).upper()
    if c[:3] in [f"O{i:02d}" for i in range(10, 17)]: return "HIPERTENSIVOS"
    if c[:3] == "O24": return "DIABETES_GEST"
    if c[:3] in ["O60", "O42", "O47"]: return "PARTO_PREMATURO"
    if c[0] == "O" and c[1] in "234": return "HEMORRAGIA"
    if c[:3] in ["O98", "O99"]: return "COMORBIDADES"
    if c[0] == "O": return "OUTROS_OBST"
    if c[:2] == "Z3": return "PRE_NATAL"
    return "OUTROS"

df["segmento"] = df["cd_cid"].apply(_seg)

n_total = len(df)
n_doc   = df["tem_qualquer"].sum()
n_evol  = df["tem_evolucao"].sum()
print(f"Gestantes total       : {n_total:,}")
print(f"Com qualquer texto    : {n_doc:,} ({n_doc/n_total*100:.1f}%)")
print(f"Com evolução médica   : {n_evol:,} ({n_evol/n_total*100:.1f}%)")
print(f"Retorno 30d (total)   : {df['y'].sum():,} ({df['y'].mean()*100:.1f}%)")
print(f"Retorno 30d (doc)     : {df[df['tem_qualquer']==1]['y'].mean()*100:.1f}%")
print(f"Retorno 30d (sem doc) : {df[df['tem_qualquer']==0]['y'].mean()*100:.1f}%")
print(f"Retorno 90d (total)   : {df['y_90d'].sum():,} ({df['y_90d'].mean()*100:.1f}%)")

## Section 2 — Documented vs Non-documented Comparison with Chi² p-values

In [ ]:
resultados = []

for seg in df["segmento"].value_counts().index:
    sub_doc = df[(df["segmento"] == seg) & (df["tem_qualquer"] == 1)]
    sub_nd  = df[(df["segmento"] == seg) & (df["tem_qualquer"] == 0)]

    if len(sub_doc) < 20 or len(sub_nd) < 20:
        continue

    ct = np.array([
        [sub_doc["y"].sum(), len(sub_doc) - sub_doc["y"].sum()],
        [sub_nd["y"].sum(),  len(sub_nd)  - sub_nd["y"].sum()],
    ])
    try:
        _, p, _, _ = scipy_stats.chi2_contingency(ct)
    except Exception:
        p = np.nan

    ret_doc = sub_doc["y"].mean() * 100
    ret_nd  = sub_nd["y"].mean() * 100
    rr = (ret_doc / ret_nd) if ret_nd > 0 else np.nan

    resultados.append({
        "segmento"        : seg,
        "n_total"         : len(sub_doc) + len(sub_nd),
        "n_doc"           : len(sub_doc),
        "n_sem_doc"       : len(sub_nd),
        "ret_doc_pct"     : round(ret_doc, 2),
        "ret_sem_doc_pct" : round(ret_nd, 2),
        "delta_pp"        : round(ret_doc - ret_nd, 2),
        "p_value"         : round(p, 4) if not np.isnan(p) else np.nan,
        "rr"              : round(rr, 3) if not np.isnan(rr) else np.nan,
    })

comp = pd.DataFrame(resultados).sort_values("ret_doc_pct")
print("\nRetorno 30d: Documentadas vs Não Documentadas por Segmento CID")
print("(delta_pp < 0 = documentadas têm MENOS retorno)")
print(comp[["segmento", "n_doc", "ret_doc_pct", "ret_sem_doc_pct", "delta_pp",
            "p_value", "rr"]].to_string(index=False))
comp.to_csv(OUT_DIR / "doc_vs_nao_doc_por_segmento.csv", index=False)

# Significance markers
def _sig(p):
    if pd.isna(p): return ""
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return ""

# Plot with significance markers
fig, ax = plt.subplots(figsize=(10, 6))
x = range(len(comp))
w = 0.35
bars_d = ax.bar([i - w/2 for i in x], comp["ret_doc_pct"],     w,
                label=T["comp_doc_lbl"],    color="steelblue", alpha=0.85)
bars_n = ax.bar([i + w/2 for i in x], comp["ret_sem_doc_pct"], w,
                label=T["comp_sem_lbl"], color="lightcoral", alpha=0.85)

# Add significance markers above each group
for i, (_, row) in enumerate(comp.iterrows()):
    sig = _sig(row["p_value"])
    if sig:
        y_top = max(row["ret_doc_pct"], row["ret_sem_doc_pct"]) + 0.5
        ax.text(i, y_top, sig, ha="center", va="bottom", fontsize=10, color="black")

ax.set_xticks(list(x))
ax.set_xticklabels(comp["segmento"], rotation=35, ha="right", fontsize=9)
ax.set_ylabel(T["comp_ylabel"])
ax.set_title(T["comp_title"])
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "01_doc_vs_nao_doc.png", dpi=150)
plt.close()
print(f"Saved: {OUT_DIR / '01_doc_vs_nao_doc.png'}")

## Section 3 — Protective Profile (within documented patients)

In [ ]:
df_doc = df[df["tem_qualquer"] == 1].copy()
print(f"\nSubgrupo documentadas: {len(df_doc):,}")
print(f"Retorno 30d neste subgrupo: {df_doc['y'].mean()*100:.1f}%")

# Reference rate
ref = df_doc["y"].mean() * 100

# Build candidate conditions
condicoes_rows = []
ig = pd.to_numeric(df_doc.get("ig_obs", df_doc.get("semanas_gestacao")), errors="coerce")

def _add_cond(nome, mask, df_sub=df_doc):
    """Compute stats for a boolean mask and append to condicoes_rows."""
    n = int(mask.sum())
    if n < 20:
        return
    sub = df_sub[mask]
    ret = sub["y"].mean() * 100
    comp_mask = ~mask
    ct = np.array([
        [sub["y"].sum(),          n - sub["y"].sum()],
        [df_sub[comp_mask]["y"].sum(),
         comp_mask.sum() - df_sub[comp_mask]["y"].sum()],
    ])
    try:
        _, p, _, _ = scipy_stats.chi2_contingency(ct)
    except Exception:
        p = np.nan
    condicoes_rows.append({
        "condicao"      : nome,
        "n"             : n,
        "ret_pct"       : round(ret, 2),
        "p_value"       : round(p, 4) if not np.isnan(p) else np.nan,
    })

# 1. Tipo de gestação
if "tipo_gestacao" in df_doc.columns:
    for tipo in ["GUTT", "GURT", "GUPT", "GGUT"]:
        mask = df_doc["tipo_gestacao"] == tipo
        _add_cond(f"tipo={tipo}", mask)

# 2. Perfil obstétrico
if "perfil_historico" in df_doc.columns:
    for perfil in df_doc["perfil_historico"].dropna().unique():
        mask = df_doc["perfil_historico"] == perfil
        _add_cond(f"perfil={perfil}", mask)

# 3. IG adequada (37-40s)
mask_ig = ig.between(37, 40, inclusive="both")
_add_cond("IG 37-40s (termo)", mask_ig)

# 4. Red flag (LLM)
if "red_flag" in df_doc.columns:
    rf = pd.to_numeric(df_doc["red_flag"], errors="coerce").fillna(-1)
    _add_cond("Sem red_flag (LLM)", rf == 0)
    _add_cond("Com red_flag (LLM)", rf == 1)

# 5. Clinical status
if "clinical_status" in df_doc.columns:
    for status in ["stable", "improving", "worsening", "critical"]:
        mask_s = df_doc["clinical_status"].fillna("") == status
        _add_cond(f"status={status}", mask_s)

# 6. Severity LLM
if "severity" in df_doc.columns:
    for sev in ["low", "moderate", "high", "critical"]:
        mask_sv = df_doc["severity"].fillna("") == sev
        _add_cond(f"severity={sev}", mask_sv)

# 7. Manchester
if "cd_cor_manchester" in df_doc.columns:
    mcr = pd.to_numeric(df_doc["cd_cor_manchester"], errors="coerce")
    _add_cond("Manchester verde/amarelo", mcr <= 2)
    _add_cond("Manchester laranja/vermelho", mcr >= 3)

# 8. Pre-natal adequado
if "nr_consultas_pre_natal" in df_doc.columns:
    nc = pd.to_numeric(df_doc["nr_consultas_pre_natal"], errors="coerce")
    _add_cond("Pré-natal ≥ 6 consultas", nc >= 6)

# 9. Primigesta + termo
_primigesta = df_doc.get(
    "primigesta", pd.Series([0] * len(df_doc), index=df_doc.index)
).fillna(0)
mask_pgt = (_primigesta == 1) & ig.reindex(df_doc.index, fill_value=np.nan).between(37, 40)
_add_cond("Primigesta + IG 37-40s", mask_pgt)

cond_df = (pd.DataFrame(condicoes_rows)
             .sort_values("ret_pct")
             .reset_index(drop=True))

cond_df["delta_vs_media"] = (cond_df["ret_pct"] - ref).round(2)
cond_df["perfil"] = cond_df["ret_pct"].apply(
    lambda v: "PROTETOR" if v < ref - 2 else ("RISCO" if v > ref + 2 else "NEUTRO")
)

# Add reference row for display
ref_row = pd.DataFrame([{
    "condicao": "─── MÉDIA GERAL ───", "n": len(df_doc),
    "ret_pct": round(ref, 2), "p_value": np.nan,
    "delta_vs_media": 0.0, "perfil": "NEUTRO",
}])
cond_df_display = pd.concat([cond_df, ref_row], ignore_index=True).sort_values("ret_pct")

print("\n" + "=" * 65)
print("PERFIS DE MENOR RISCO — Pacientes Documentadas")
print(f"Referência (média geral): {ref:.1f}%")
print("=" * 65)
print(cond_df_display[["condicao", "n", "ret_pct", "delta_vs_media", "perfil",
                        "p_value"]].to_string(index=False))
cond_df.to_csv(OUT_DIR / "perfil_protetor_documentadas.csv", index=False)

# Plot
fig, ax = plt.subplots(figsize=(10, max(6, len(cond_df_display) * 0.4)))
cores = {"PROTETOR": "#2ca02c", "NEUTRO": "steelblue", "RISCO": "#d62728"}
for i, (_, row) in enumerate(cond_df_display.iterrows()):
    cor = cores.get(row["perfil"], "gray")
    ax.barh(row["condicao"], row["ret_pct"], color=cor, alpha=0.85)
    sig = _sig(row.get("p_value", np.nan))
    if sig:
        ax.text(row["ret_pct"] + 0.1, i, sig, va="center", fontsize=9)
ax.axvline(ref, color="black", ls="--", lw=1.5, label=T["perfil_media_fmt"].format(ref=ref))
ax.set_xlabel(T["perfil_xlabel"])
ax.set_title(T["perfil_title"])
ax.legend(fontsize=9)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "02_perfil_protetor.png", dpi=150)
plt.close()
print(f"Saved: {OUT_DIR / '02_perfil_protetor.png'}")

## Section 4 — Adjusted Odds Ratio for Protective Conditions

In [ ]:
import statsmodels.api as sm
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Identify protective conditions eligible for adjusted OR
protetor_conds = cond_df[
    (cond_df["perfil"] == "PROTETOR") &
    (cond_df["n"] >= 50) &
    (cond_df["p_value"].fillna(1) < 0.2)
].copy()

print(f"\nCondições protetoras elegíveis para OR ajustado: {len(protetor_conds)}")
print(protetor_conds[["condicao", "n", "ret_pct", "p_value"]].to_string(index=False))

# Confounders available in the data
CONFOUNDERS = [c for c in ["idade_anos", "vl_escore_triagem"] if c in df_doc.columns]
CAT_CONFOUNDERS = [c for c in ["segmento", "tp_atendimento"] if c in df_doc.columns]

or_rows = []

for _, cond_row in protetor_conds.iterrows():
    nome = cond_row["condicao"]

    # Rebuild the binary exposure T for this condition
    try:
        # Match condition masks defined in Section 3
        if nome.startswith("tipo="):
            tipo = nome.split("=", 1)[1]
            T = (df_doc["tipo_gestacao"] == tipo).astype(int)
        elif nome.startswith("perfil="):
            pf = nome.split("=", 1)[1]
            T = (df_doc["perfil_historico"].fillna("") == pf).astype(int)
        elif nome == "IG 37-40s (termo)":
            T = ig.between(37, 40, inclusive="both").astype(int)
        elif nome == "Sem red_flag (LLM)":
            rf = pd.to_numeric(df_doc["red_flag"], errors="coerce").fillna(-1)
            T = (rf == 0).astype(int)
        elif nome.startswith("status="):
            st = nome.split("=", 1)[1]
            T = (df_doc["clinical_status"].fillna("") == st).astype(int)
        elif nome.startswith("severity="):
            sv = nome.split("=", 1)[1]
            T = (df_doc["severity"].fillna("") == sv).astype(int)
        elif nome == "Manchester verde/amarelo":
            mcr = pd.to_numeric(df_doc["cd_cor_manchester"], errors="coerce")
            T = (mcr <= 2).astype(int)
        elif nome == "Pré-natal ≥ 6 consultas":
            nc = pd.to_numeric(df_doc["nr_consultas_pre_natal"], errors="coerce")
            T = (nc >= 6).astype(int)
        elif nome == "Primigesta + IG 37-40s":
            _pg = df_doc.get("primigesta",
                              pd.Series([0] * len(df_doc), index=df_doc.index)).fillna(0)
            T = ((_pg == 1) & ig.reindex(df_doc.index, fill_value=np.nan).between(37, 40)).astype(int)
        else:
            print(f"  SKIP (mask not reconstructed): {nome}")
            continue
    except Exception as ex:
        print(f"  SKIP {nome}: {ex}")
        continue

    # Build design matrix — MICE imputation for numeric confounders
    X_parts = [T.rename("exposure").astype(float).reset_index(drop=True)]
    if CONFOUNDERS:
        num_conf_mat = np.column_stack([
            pd.to_numeric(df_doc[cf], errors="coerce").values for cf in CONFOUNDERS
        ])
        _mice = IterativeImputer(max_iter=10, random_state=42, min_value=0)
        num_conf_imputed = _mice.fit_transform(num_conf_mat)
        for j, cf in enumerate(CONFOUNDERS):
            X_parts.append(
                pd.Series(num_conf_imputed[:, j], name=cf)
            )

    for cf in CAT_CONFOUNDERS:
        dummies = pd.get_dummies(df_doc[cf].fillna("MISSING"), prefix=cf, drop_first=True)
        dummies = dummies.reset_index(drop=True)
        X_parts.append(dummies)

    X_reg = pd.concat(X_parts, axis=1).astype(float)
    y_reg = df_doc["y"].values

    # Standardise numeric confounders (not the binary exposure)
    num_cols_reg = CONFOUNDERS
    for col in num_cols_reg:
        if col in X_reg.columns:
            X_reg[col] = (X_reg[col] - X_reg[col].mean()) / (X_reg[col].std() + 1e-9)

    # Try statsmodels first, fall back to sklearn
    try:
        X_sm = sm.add_constant(X_reg.values.astype(float))
        logit = sm.Logit(y_reg, X_sm)
        result = logit.fit(disp=0, maxiter=200)
        coef  = result.params[1]       # exposure coefficient
        se    = result.bse[1]
        p_val = result.pvalues[1]
        or_   = np.exp(coef)
        ci_lo = np.exp(coef - 1.96 * se)
        ci_hi = np.exp(coef + 1.96 * se)
        method = "statsmodels"
    except Exception as e_sm:
        print(f"  statsmodels falhou ({e_sm}), usando sklearn bootstrap para {nome}")
        try:
            from sklearn.utils import resample
            lr = LogisticRegression(max_iter=500, solver="lbfgs", random_state=42)
            lr.fit(X_reg.values, y_reg)
            coef = lr.coef_[0][0]      # exposure is first column
            or_  = np.exp(coef)
            # Bootstrap CI (100 iterations for speed)
            boot_coefs = []
            for _ in range(100):
                Xb, yb = resample(X_reg.values, y_reg, random_state=None)
                try:
                    lr2 = LogisticRegression(max_iter=500, solver="lbfgs", random_state=42)
                    lr2.fit(Xb, yb)
                    boot_coefs.append(lr2.coef_[0][0])
                except Exception:
                    pass
            ci_lo = np.exp(np.percentile(boot_coefs, 2.5))  if boot_coefs else np.nan
            ci_hi = np.exp(np.percentile(boot_coefs, 97.5)) if boot_coefs else np.nan
            p_val = np.nan
            method = "sklearn_bootstrap"
        except Exception as e_sk:
            print(f"  sklearn também falhou: {e_sk}")
            continue

    or_rows.append({
        "condicao"  : nome,
        "n"         : int(T.sum()),
        "or_ajustado": round(or_, 3),
        "ci_lo"     : round(ci_lo, 3) if not np.isnan(ci_lo) else np.nan,
        "ci_hi"     : round(ci_hi, 3) if not np.isnan(ci_hi) else np.nan,
        "p_ajustado": round(p_val, 4) if not np.isnan(p_val) else np.nan,
        "metodo"    : method,
    })
    print(f"  {nome}: OR={or_:.3f} [95% CI {ci_lo:.3f}–{ci_hi:.3f}]  p={p_val:.4f}")

or_df = pd.DataFrame(or_rows)
if not or_df.empty:
    or_df.to_csv(OUT_DIR / "or_ajustado_condicoes_protetoras.csv", index=False)
    print(f"\nOR ajustado salvo: {OUT_DIR / 'or_ajustado_condicoes_protetoras.csv'}")

    # Forest plot
    fig, ax = plt.subplots(figsize=(9, max(4, len(or_df) * 0.7)))
    ys = range(len(or_df))
    ax.errorbar(
        or_df["or_ajustado"], ys,
        xerr=[or_df["or_ajustado"] - or_df["ci_lo"].fillna(or_df["or_ajustado"]),
              or_df["ci_hi"].fillna(or_df["or_ajustado"]) - or_df["or_ajustado"]],
        fmt="o", color="steelblue", ecolor="gray", elinewidth=2, capsize=4, markersize=7
    )
    ax.axvline(1.0, color="black", ls="--", lw=1.2, label="OR = 1")
    ax.set_yticks(list(ys))
    ax.set_yticklabels(or_df["condicao"], fontsize=9)
    ax.set_xlabel(T["or_xlabel"])
    ax.set_title(T["or_title"])
    ax.legend(fontsize=9)
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "03_or_ajustado.png", dpi=150)
    plt.close()
    print(f"Saved: {OUT_DIR / '03_or_ajustado.png'}")
else:
    print("Nenhuma condição protetora com dados suficientes para OR ajustado.")

## Section 5 — Protective Combinations

In [ ]:
print("\n" + "=" * 65)
print("COMBINAÇÕES PROTETORAS — menor risco com documentação")
print("=" * 65)

combos = {
    "GUTT + sem red_flag":
        (df_doc.get("tipo_gestacao", pd.Series([""] * len(df_doc), index=df_doc.index)) == "GUTT") &
        (pd.to_numeric(df_doc.get("red_flag", 0), errors="coerce").fillna(1) == 0),

    "Primigesta + IG 37-40s + sem red_flag":
        (df_doc.get("primigesta", pd.Series([0] * len(df_doc), index=df_doc.index)).fillna(0) == 1) &
        (pd.to_numeric(df_doc.get("ig_obs", df_doc.get("semanas_gestacao", np.nan)),
                       errors="coerce").between(37, 40)) &
        (pd.to_numeric(df_doc.get("red_flag", 0), errors="coerce").fillna(1) == 0),

    "Manchester ≤ 2 + sem red_flag + status=stable":
        (pd.to_numeric(df_doc.get("cd_cor_manchester", 99), errors="coerce").fillna(99) <= 2) &
        (pd.to_numeric(df_doc.get("red_flag", 0), errors="coerce").fillna(1) == 0) &
        (df_doc.get("clinical_status", pd.Series([""] * len(df_doc), index=df_doc.index)).fillna("") == "stable"),

    "MULTIPARA_SEM_PERDA + termo + Manchester baixo":
        (df_doc.get("perfil_historico", pd.Series([""] * len(df_doc), index=df_doc.index)).fillna("") == "MULTIPARA_SEM_PERDA") &
        (pd.to_numeric(df_doc.get("semanas_gestacao", 0), errors="coerce").fillna(0) >= 37) &
        (pd.to_numeric(df_doc.get("cd_cor_manchester", 99), errors="coerce").fillna(99) <= 2),

    "Documentação rica (≥3 evoluções) + severity=low":
        (df_doc.get("qtd_evolucoes_documentadas",
                    pd.Series([0] * len(df_doc), index=df_doc.index)).fillna(0) >= 3) &
        (df_doc.get("severity", pd.Series([""] * len(df_doc), index=df_doc.index)).fillna("") == "low"),
}

rows_combo = []
for nome, mask in combos.items():
    try:
        sub_c = df_doc[mask.values if hasattr(mask, "values") else mask]
        if len(sub_c) < 10:
            print(f"  {nome}: n={len(sub_c)} — insuficiente")
            continue
        ret = sub_c["y"].mean() * 100
        # Chi² vs rest of documented
        rest = df_doc[~(mask.values if hasattr(mask, "values") else mask)]
        ct = np.array([
            [sub_c["y"].sum(),  len(sub_c)  - sub_c["y"].sum()],
            [rest["y"].sum(),   len(rest)   - rest["y"].sum()],
        ])
        try:
            _, p_combo, _, _ = scipy_stats.chi2_contingency(ct)
        except Exception:
            p_combo = np.nan

        rows_combo.append({
            "combinacao"     : nome,
            "n"              : len(sub_c),
            "ret_pct"        : round(ret, 2),
            "delta_vs_media" : round(ret - ref, 2),
            "p_value"        : round(p_combo, 4) if not np.isnan(p_combo) else np.nan,
        })
        sinal = "PROTETOR" if ret < ref - 2 else ("RISCO" if ret > ref + 2 else "~neutro")
        print(f"  {sinal}  {nome}")
        print(f"          n={len(sub_c):,}  retorno={ret:.1f}%  (média={ref:.1f}%)  "
              f"p={p_combo:.4f}" if not np.isnan(p_combo) else
              f"          n={len(sub_c):,}  retorno={ret:.1f}%  (média={ref:.1f}%)")
    except Exception as e:
        print(f"  SKIP {nome}: {e}")

if rows_combo:
    combo_df = pd.DataFrame(rows_combo).sort_values("ret_pct")
    combo_df.to_csv(OUT_DIR / "combinacoes_protetoras.csv", index=False)

    fig, ax = plt.subplots(figsize=(9, max(4, len(combo_df) * 0.8)))
    cores_c = ["#2ca02c" if v < -2 else "#d62728" if v > 2 else "steelblue"
               for v in combo_df["delta_vs_media"]]
    ax.barh(combo_df["combinacao"][::-1], combo_df["ret_pct"][::-1], color=cores_c[::-1])
    ax.axvline(ref, color="black", ls="--", lw=1.5, label=T["comb_media_fmt"].format(ref=ref))
    ax.set_xlabel(T["comb_xlabel"])
    ax.set_title(T["comb_title"])
    ax.legend(fontsize=9)
    ax.grid(axis="x", alpha=0.3)
    for i, (_, row) in enumerate(combo_df[::-1].reset_index(drop=True).iterrows()):
        sig = _sig(row.get("p_value", np.nan))
        ax.text(row["ret_pct"] + 0.2, i,
                f"n={int(row['n']):,}  Δ={row['delta_vs_media']:+.1f}pp {sig}",
                va="center", fontsize=8)
    plt.tight_layout()
    plt.savefig(OUT_DIR / "04_combinacoes_protetoras.png", dpi=150)
    plt.close()
    print(f"Saved: {OUT_DIR / '04_combinacoes_protetoras.png'}")

## Section 6 — LLM Feature Coverage Report

In [ ]:
LLM_FEATURES = ["red_flag", "clinical_status", "severity_score", "clinical_status_score"]
llm_cov_rows = []

for feat in LLM_FEATURES:
    if feat not in df.columns:
        continue
    col = df[feat]
    n_total_f = len(col)
    n_present = col.notna().sum()
    # For categorical string features, numeric coercion may fail
    col_num = pd.to_numeric(col, errors="coerce")
    coverage_pct = round(n_present / n_total_f * 100, 2)
    mean_val     = round(col_num.mean(), 4) if col_num.notna().any() else np.nan
    median_val   = round(col_num.median(), 4) if col_num.notna().any() else np.nan
    try:
        spearman_r, spearman_p = scipy_stats.spearmanr(
            col_num.fillna(col_num.median()), df["y"], nan_policy="omit"
        )
        spearman_r = round(spearman_r, 4)
        spearman_p = round(spearman_p, 6)
    except Exception:
        spearman_r, spearman_p = np.nan, np.nan

    llm_cov_rows.append({
        "feature"      : feat,
        "n_present"    : int(n_present),
        "coverage_pct" : coverage_pct,
        "mean"         : mean_val,
        "median"       : median_val,
        "spearman_r"   : spearman_r,
        "spearman_p"   : spearman_p,
    })

llm_cov_df = pd.DataFrame(llm_cov_rows)
if not llm_cov_df.empty:
    print("\nLLM Feature Coverage")
    print(llm_cov_df.to_string(index=False))
    llm_cov_df.to_csv(OUT_DIR / "llm_cobertura_08.csv", index=False)
    print(f"Saved: {OUT_DIR / 'llm_cobertura_08.csv'}")
else:
    print("\nNenhuma feature LLM encontrada no dataset.")

## Section 7 — Save Shared Artifacts

In [ ]:
# Determine FEATURES_FOR_SHAP (mirrors original feature list)
FEATURES_FOR_SHAP = [f for f in [
    "segmento", "tipo_gestacao", "perfil_historico",
    "idade_anos", "ig_obs", "semanas_gestacao",
    "vl_escore_triagem", "cd_cor_manchester",
    "qtd_pre_med", "qtd_evolucoes_documentadas",
    "duracao_horas_aprox", "media_peso_rn_g",
    "risco_gestacao", "gestacao_pretermo", "primigesta",
    "grande_multipara_v2", "tem_aborto_v2", "aborto_recorrente",
    "red_flag", "severity_score", "clinical_status_score",
] if f in df_doc.columns]

# Top 3 protective conditions for temporal stability in 08.2
top3_protetores = (
    cond_df[cond_df["perfil"] == "PROTETOR"]
    .sort_values("ret_pct")
    .head(3)["condicao"]
    .tolist()
)

ART_DIR.mkdir(parents=True, exist_ok=True)
with open(ART_DIR / "df_doc.pkl", "wb") as f:
    pickle.dump({
        "df"             : df,
        "df_doc"         : df_doc,
        "ref"            : ref,
        "FEATURES"       : FEATURES_FOR_SHAP,
        "top3_protetores": top3_protetores,
        "cond_df"        : cond_df,
    }, f)
print(f"[08.1] Artefatos salvos: {ART_DIR}")

## Summary

In [ ]:
print()
print("╔══ 08.1 Perfil de Menor Risco — Documentadas — Resumo ══╗")
print(f"  N total               : {n_total:,}")
print(f"  N documentadas        : {len(df_doc):,}")
print(f"  Taxa retorno 30d (doc): {ref:.1f}%")
n_prot = len(cond_df[cond_df['perfil'] == 'PROTETOR'])
n_risk = len(cond_df[cond_df['perfil'] == 'RISCO'])
print(f"  Condições protetoras  : {n_prot}")
print(f"  Condições de risco    : {n_risk}")
print(f"  OR ajustados gerados  : {len(or_df) if 'or_df' in dir() else 0}")
print(f"  Saída                 : {OUT_DIR}")
print(f"  Artefatos             : {ART_DIR}")
print("╚═════════════════════════════════════════════════════════╝")
print(f"\nConcluído — {datetime.now().isoformat()}")